### RunnableBranch

In [1]:
!uv --version

uv 0.12.17 (635500036 2026-09-18 x86_64-pc-windows-msvc)


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_teddynote import logging

logging.langsmith("LCEL-Advanced")

LangSmith 추적을 시작합니다.
[프로젝트명]
LCEL-Advanced


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """주어진 사용자 질문을 '수학', '과학', 또는 '기타' 중 하나로 분류하세요. 한 단어 이상으로 응답하지 마세요.
    
    <question>
    {question}
    </question>
    
    Classification:"""
)

chain = (
    prompt
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

In [ ]:
chain.invoke({"question": "2+2 는 무엇인가요?"})

'수학'

Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError(\'HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Max retries exceeded with url: /info (Caused by NameResolutionError("HTTPSConnection(host=\\\'api.smith.langchain.com\\\', port=443): Failed to resolve \\\'api.smith.langchain.com\\\' ([Errno 11002] getaddrinfo failed)"))\'))\nContent-Length: None\nAPI Key: lsv2_********************************************7a')
Run compression is not enabled. Please update to the latest version of LangSmith. Falling back to regular multipart ingestion.


In [6]:
chain.invoke({"question": "작용 반작용의 법칙은 무엇인가요?"})

'과학'

In [7]:
chain.invoke({"question": "Google은 어떤 회사인가요?"})

'기타'

In [8]:
math_chain = (
    PromptTemplate.from_template(
        """You are an expert in math. \
            Always answer questions starting with "깨봉선생님께서 말씀하시기를..". \
            Respond to the following question:
            
            Question: {question}
            Answer:"""
    )
    | ChatOpenAI(model="gpt-4o-mini")
)

science_chain = (
    PromptTemplate.from_template(
        """You are an expert in science. \
            Always answer question starting with "아이작 뉴턴이 말씀하시길..". \
            Respond to the following question:
            
            Question: {question}
            Answer:"""
    )
    | ChatOpenAI(model="gpt-4o-mini")
)

general_chain = (
    PromptTemplate.from_template(
        """Respond to the follwing question concisely:
        
        Question: {question}
        Answer:"""
    )
    | ChatOpenAI(model="gpt-4o-mini")
)

In [9]:
def route(info):
    if "수학" in info["topic"].lower():
        return math_chain

    elif "과학" in info["topic"].lower():
        return science_chain

    else:
        return general_chain

In [10]:
from operator import itemgetter
from langchain_core.runnables import RunnableLambda

full_chain = (
    {"topic": chain, "question": itemgetter("question")}
    | RunnableLambda(
        route
    )
    | StrOutputParser()
)

In [11]:
full_chain.invoke({"question": "미적분의 개념에 대해 말씀해 주세요"})

'깨봉선생님께서 말씀하시기를, 미적분은 수학의 한 분야로, 주로 함수의 변화와 기하학적 양을 다루는 도구입니다. 미적분은 크게 두 가지 중심 개념으로 나뉘어 있습니다: 미분과 적분입니다.\n\n미분은 함수의 기울기나 변화율을 설명하며, 특정 점에서 함수의 순간적인 변화량을 측정합니다. 예를 들어, 자동차의 속도를 구하는 것은 해당 자동차의 위치 함수의 미분을 통해 이루어질 수 있습니다.\n\n반면, 적분은 함수의 총합적 양이나 면적을 구하는 방법으로, 연속적인 점의 집합에 대한 합계를 나타냅니다. 그래픽적으로는 곡선 아래의 면적을 계산하는 데 사용됩니다.\n\n미적분학은 물리학, 공학, 경제학 등 다양한 분야에서 실제 문제를 해결하는 데 필수적인 도구로 활용됩니다.'

In [12]:
full_chain.invoke({"question": "중력은 어떻게 작용하나요?"})

'아이작 뉴턴이 말씀하시길, 중력은 두 물체 간의 질량에 비례하고, 두 물체 간의 거리의 제곱에 반비례하는 힘이라고 하였습니다. 즉, 질량이 큰 물체는 더 강한 중력을 가지고 있으며, 물체 간의 거리가 가까울수록 그 힘은 더욱 강해진다는 것입니다. 뉴턴의 만유인력 법칙에 따르면, 모든 물체는 서로 중력으로 끌어당기며, 이러한 힘이 우주에서의 행성과 별들의 운동을 지배하고 있습니다. 따라서, 중력은 우리가 지구에 서 있을 수 있게 하고, 천체들이 궤도를 이루어 움직이도록 만드는 중요한 힘입니다.'

In [13]:
full_chain.invoke({"question": "RAG은 무엇이가요?"})

'RAG은 "Retrieval-Augmented Generation"의 약자로, 정보 검색과 생성 모델을 결합한 방식입니다. 이 방법은 외부 데이터베이스에서 정보를 검색한 후, 이를 바탕으로 보다 정확하고 관련성 높은 응답을 생성하는 데 사용됩니다.'

In [14]:
from operator import itemgetter
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: "수학" in x["topic"].lower(), math_chain),
    (lambda x: "과학" in x["topic"].lower(), science_chain),
    general_chain,
)

full_chain = (
    {"topic": chain, "question": itemgetter("question")} | branch | StrOutputParser()
)

In [15]:
full_chain.invoke({"question": "미적분의 개념에 대해 알려줘"})

'깨봉선생님께서 말씀하시기를, 미적분은 수학의 한 분야로, 함수의 변화와 관련된 개념을 다룹니다. 미적분은 기본적으로 두 가지 주요 개념인 미분과 적분으로 나눌 수 있습니다. \n\n미분은 함수의 변화율, 즉 어떤 함수가 특정한 점에서 얼마나 빠르게 변하는지를 측정합니다. 이는 기울기의 개념과 밀접하게 연결되어 있습니다. 예를 들어, 자동차의 속도를 알고 싶을 때, 시간을 기준으로 거리의 변화량을 알아보는 것이 미분입니다.\n\n적분은 미분과 반대의 개념으로, 함수의 특정 구간에 대한 전체적인 성질을 파악하는 데 사용됩니다. 주로 면적이나 부피를 계산할 때 쓰이며, 이것은 무한히 작은 조각들을 합치는 과정으로 이해할 수 있습니다.\n\n미적분은 물리학, 공학, 경제학 등 다양한 분야에서 응용되며, 실생활에서도 매우 중요한 역할을 합니다. 이러한 개념들을 잘 이해하면, 자연 현상이나 다양한 문제를 더 정확하게 분석하고 해결할 수 있게 됩니다.'

In [16]:
full_chain.invoke({"question": "중력 가속도 어떻게 계산해?"})

'아이작 뉴턴이 말씀하시길, 중력 가속도는 지구의 중력 영향을 받는 물체에 작용하는 힘을 통해 계산할 수 있습니다. 이 힘은 물체의 질량과 지구의 질량 사이의 인력으로 설명되며, 중력이 가속도로 나타나는 방식은 뉴턴의 법칙에 따라 F=ma (힘 = 질량 × 가속도)로 표현됩니다. \n\n중력 가속도 g는 일반적으로 지구 표면에서 약 9.81 m/s²로 정의됩니다. 이를 구하는 방법은 다음과 같습니다:\n\n1. 지구의 질량(M)과 반지름(R)을 알고 있어야 합니다.\n2. 중력의 법칙에 따라 g = G * M / R²를 사용하여 계산할 수 있으며, 여기서 G는 만유인력 상수입니다 (약 6.674×10^-11 N(m/kg)²).\n\n따라서, 이러한 데이터를 통해 지구에서의 중력 가속도를 계산할 수 있습니다.'

In [17]:
full_chain.invoke({"question": "RAG은 무엇인가?"})

'RAG( retrieval-augmented generation )은 정보 검색과 자연어 생성 기술을 결합한 모델로, 외부 데이터베이스에서 정보를 검색한 후 이를 기반으로 텍스트를 생성하는 방식입니다.'